# Infinite DOM — Training Notebook

**Runtime:** Colab T4 (smoke test) or A100 (full training)

## Pipeline
1. Install dependencies & configure remote environment
2. Load oracle training data (with CoT reasoning augmentation)
3. SFT warmup with **Think-then-Act** format (WebAgent-R1 style)
4. GRPO RL with reasoning-aware reward + task curriculum
5. Live environment evaluation with step history (dynamic context compression)
6. Save model + plots

## Research-Backed Training Design

### From WebAgent-R1 (SOTA web agent training):
- **Think-then-Act format**: `<think>reasoning</think><answer>action</answer>` — their single biggest improvement
- **Long-CoT data augmentation**: Enrich oracle trajectories with detailed reasoning traces for SFT
- **Dynamic context compression**: Compress step history to maintain multi-turn context
- **Behaviour cloning (SFT) is CRUCIAL before RL**: R1-Zero (no SFT) actually degraded performance

### From WebRL (self-evolving curriculum):
- **Multi-component reward scoring**: Score both reasoning quality AND action correctness
- **Task curriculum**: Progressive difficulty (clean → drift → chaos)

### Training Quality Safeguards:
- **Anti-overfitting**: Early stopping, stratified split, LoRA capacity limit
- **Anti-reward-hacking**: Local oracle-comparison + reasoning quality scoring
- **Anti-underfitting**: Sufficient data, sanity checks, curriculum progression
- **Anti-mode-collapse**: Diversity monitoring, temperature sampling

In [2]:
# Cell 1 — Install dependencies
!pip install -q unsloth
!pip install -q "trl>=0.12.0" transformers accelerate peft
!pip install -q httpx pydantic datasets matplotlib
!pip install -q huggingface_hub

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 14:
        print("WARNING: Less than 14GB VRAM — T4 (16GB) or better recommended")
else:
    print("WARNING: No GPU detected — training will be extremely slow")

GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Cell 2 — Configure Remote Environment & Health Check
#
# Your HF Space must be running before you start training.
# The notebook talks to the environment over HTTP for evaluation only.
# SFT and GRPO training use LOCAL oracle data — no live env needed during training.

import os
import time
import httpx

# ============================================================
# CONFIGURE YOUR HF SPACE URL HERE
# ============================================================
HF_SPACE_URL = "https://saksham1771-infinite-dom.hf.space"  # @param {type:"string"}
# ============================================================

# Strip trailing slash
HF_SPACE_URL = HF_SPACE_URL.rstrip("/")
os.environ["INFINITE_DOM_URL"] = HF_SPACE_URL

print(f"Environment URL: {HF_SPACE_URL}")
print()

# Health check with retry
def check_health(url, max_retries=3, initial_wait=5):
    """Verify the remote environment is reachable."""
    for attempt in range(max_retries):
        try:
            r = httpx.get(f"{url}/health", timeout=30)
            if r.status_code == 200:
                data = r.json()
                print(f"Connected! Server status: {data}")
                return True
            else:
                print(f"  Attempt {attempt+1}: HTTP {r.status_code}")
        except httpx.ConnectError:
            print(f"  Attempt {attempt+1}: Connection refused — is the HF Space running?")
        except httpx.TimeoutException:
            print(f"  Attempt {attempt+1}: Timeout — Space may be cold-starting (takes 1-2 min)")
        except Exception as e:
            print(f"  Attempt {attempt+1}: {type(e).__name__}: {e}")

        if attempt < max_retries - 1:
            wait = initial_wait * (2 ** attempt)
            print(f"  Retrying in {wait}s...")
            time.sleep(wait)

    print()
    print("FAILED to connect to HF Space.")
    print("Possible fixes:")
    print("  1. Check your HF_SPACE_URL is correct")
    print("  2. Make sure the Space is running (not sleeping)")
    print("  3. Visit the Space URL in your browser to wake it up")
    print()
    print("NOTE: Training (SFT + GRPO) does NOT need the live environment.")
    print("      Only Cell 10 (live evaluation) requires the connection.")
    print("      You can train now and evaluate later when the Space is ready.")
    return False

env_available = check_health(HF_SPACE_URL)
if env_available:
    # Fetch available tasks
    try:
        r = httpx.get(f"{HF_SPACE_URL}/tasks", timeout=10)
        if r.status_code == 200:
            tasks = r.json().get("tasks", [])
            print(f"\nAvailable tasks ({len(tasks)}):")
            for t in tasks:
                print(f"  Task {t['task_id']}: {t['description']}")
    except Exception:
        pass

Environment URL: https://saksham1771-infinite-dom.hf.space

Connected! Server status: {'status': 'healthy'}

Available tasks (4):
  Task 1: Clean booking form — standard labels, no distractors
  Task 2: Label drift — randomised field labels and button text
  Task 3: Structural drift — randomised layout and field order
  Task 4: Full chaos — distractors, noisy ARIA, everything randomised


In [8]:
# Cell 3 — Load Oracle Training Data
#
# Downloads from HuggingFace Hub (default) or uses local files.
# Downloads BOTH files:
#   - oracle_trajectories.jsonl (raw oracle data)
#   - oracle_cot.jsonl (CoT-augmented data for Think-then-Act SFT)
#
# Cell 4 will automatically use the CoT version if downloaded.

import json
from pathlib import Path
from collections import Counter

# ============================================================
# CONFIGURE DATA SOURCE
# ============================================================
DATA_SOURCE = "huggingface"  # @param ["huggingface", "local"]
HF_DATASET_REPO = "saksham1771/infinite-dom-data"  # @param {type:"string"}
LOCAL_DATA_PATH = "training/data/oracle_trajectories.jsonl"  # @param {type:"string"}
# ============================================================

COT_DATA_PATH = Path("training/data/oracle_cot.jsonl")  # Cell 4 checks this path

if DATA_SOURCE == "huggingface":
    from huggingface_hub import hf_hub_download

    # Download raw oracle data
    local_file = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename="oracle_trajectories.jsonl",
        repo_type="dataset",
    )
    LOCAL_DATA_PATH = local_file
    print(f"Downloaded oracle_trajectories.jsonl from {HF_DATASET_REPO}")

    # Download CoT-augmented data (if it exists in the repo)
    try:
        cot_local_file = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename="oracle_cot.jsonl",
            repo_type="dataset",
        )
        # Copy to the path Cell 4 expects
        COT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(cot_local_file, str(COT_DATA_PATH))
        print(f"Downloaded oracle_cot.jsonl from {HF_DATASET_REPO}")
        print(f"  Saved to {COT_DATA_PATH} (Cell 4 will use this for Think-then-Act SFT)")
    except Exception as e:
        print(f"No oracle_cot.jsonl in dataset repo ({e})")
        print("  Cell 4 will fall back to raw oracle data (no reasoning traces)")
        print("  To fix: run augment_cot.py locally, then upload oracle_cot.jsonl to your dataset repo")

with open(LOCAL_DATA_PATH) as f:
    records = [json.loads(line) for line in f if line.strip()]

# Validate data integrity
required_keys = {"task_id", "seed", "step", "instruction", "observation", "action"}
valid_records = []
invalid_count = 0
for r in records:
    if required_keys.issubset(r.keys()):
        action = r["action"]
        if action.get("action_type") in ("click", "type", "scroll", "wait"):
            valid_records.append(r)
        else:
            invalid_count += 1
    else:
        invalid_count += 1

records = valid_records
if invalid_count > 0:
    print(f"WARNING: Dropped {invalid_count} invalid records")

# Statistics
task_counts = Counter(r["task_id"] for r in records)
action_type_counts = Counter(r["action"]["action_type"] for r in records)
avg_obs_len = sum(len(r["observation"]) for r in records) / max(len(records), 1)

print(f"\nLoaded {len(records)} valid oracle records")
print(f"\nRecords per task:")
for tid in sorted(task_counts):
    print(f"  Task {tid}: {task_counts[tid]} records")
print(f"\nAction type distribution:")
for atype, count in action_type_counts.most_common():
    print(f"  {atype}: {count} ({100*count/len(records):.1f}%)")
print(f"\nAvg observation length: {avg_obs_len:.0f} chars")
print(f"CoT data available: {'Yes' if COT_DATA_PATH.exists() else 'No'}")

Downloaded oracle_trajectories.jsonl from saksham1771/infinite-dom-data
Downloaded oracle_cot.jsonl from saksham1771/infinite-dom-data
  Saved to training/data/oracle_cot.jsonl (Cell 4 will use this for Think-then-Act SFT)

Loaded 250 valid oracle records

Records per task:
  Task 1: 125 records
  Task 2: 125 records

Action type distribution:
  click: 222 (88.8%)
  type: 28 (11.2%)

Avg observation length: 2565 chars
CoT data available: Yes


In [10]:
# Cell 4 — Prepare SFT Dataset with Think-then-Act Format (WebAgent-R1)
#
# KEY CHANGE: Instead of training on raw "observation → JSON action", we train
# on "observation → <think>reasoning</think><answer>JSON action</answer>"
#
# WebAgent-R1 showed this is the single biggest performance driver for web agents.
# The model learns to REASON about what it sees before deciding what to do.
#
# If CoT-augmented data exists (from augment_cot.py), use it.
# Otherwise, fall back to raw oracle data with plain JSON responses.
#
# We pre-format texts here so Cell 6 doesn't need a formatting_func
# (avoids HuggingFace Datasets serialization issues with nested dicts).

import re
from pathlib import Path
from datasets import Dataset, DatasetDict

# --- Added to resolve NameError: 'tokenizer' is not defined ---
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"  # Small enough for T4, capable enough for tasks
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # Auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit=True, # QLoRA — fits in T4 16GB
)
# ------------------------------------------------------------

# --- Think-then-Act System Prompt (WebAgent-R1 style) ---
SYSTEM_PROMPT = """You are a web agent navigating an interactive web application.
You observe an accessibility tree and must complete a booking task.

First, reason about what you see and what action to take inside <think> tags.
Then, provide your action as a JSON object inside <answer> tags.

Format:
<think>
[Observe the current page state. Identify which fields are filled, what buttons are available, and what the next logical step is to complete the task.]
</think>
<answer>
{"action_type": "click"|"type"|"scroll"|"wait", "element_ref": "ref_id", "text_value": "text"|"", "scroll_delta": 0}
</answer>

Rules:
- Use "type" to fill text fields (element_ref + text_value required)
- Use "click" to press buttons/links (element_ref required)
- Use "scroll" to scroll the page (scroll_delta: positive=down, negative=up)
- Use "wait" when the page is loading or you need to observe
- Element refs look like: inp_1, btn_2, cmb_1, lnk_3
- Always dismiss cookie banners or popups before interacting with the main form"""

OBS_MAX_CHARS = 3500

# --- Try loading CoT-augmented data first ---
COT_DATA_PATH = Path("training/data/oracle_cot.jsonl")
USE_COT = COT_DATA_PATH.exists()

if USE_COT:
    print("Found CoT-augmented data — using Think-then-Act format for SFT")
    with open(COT_DATA_PATH) as f:
        cot_records = [json.loads(line) for line in f if line.strip()]
    print(f"  Loaded {len(cot_records)} CoT-augmented records")
else:
    print("No CoT data found. Run: python training/augment_cot.py")
    print("Falling back to raw oracle data (no reasoning traces)")
    cot_records = None


def format_for_sft(record):
    """Convert an oracle record into a pre-formatted text string for SFT."""
    obs_text = record["observation"][:OBS_MAX_CHARS]

    step = record.get("step", 0)
    step_ctx = f"\nStep: {step}" if step > 0 else ""

    user = f"Task: {record['instruction']}\n\nAccessibility Tree:\n{obs_text}{step_ctx}"

    # Clean action dict
    action = record["action"].copy()
    for key in list(action.keys()):
        if key not in ("action_type", "element_ref", "text_value", "scroll_delta"):
            del action[key]
    action.setdefault("scroll_delta", 0)
    action.setdefault("text_value", "")
    action.setdefault("element_ref", "")

    # Use CoT response if available, otherwise plain JSON
    if USE_COT and "cot_response" in record:
        assistant_text = record["cot_response"]
    else:
        assistant_text = json.dumps(action, separators=(",", ":"))

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_text},
    ]

    # Pre-format into a single text string using the tokenizer's chat template
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    return {
        "text": text,
        "task_id": record["task_id"],
    }


source_records = cot_records if USE_COT else records
sft_data = [format_for_sft(r) for r in source_records]

# Stratified train/eval split — ensures all task levels in eval
from collections import defaultdict
import random as stdlib_random

stdlib_random.seed(42)
by_task = defaultdict(list)
for item in sft_data:
    by_task[item["task_id"]].append(item)

train_items, eval_items = [], []
for task_id in sorted(by_task.keys()):
    items = by_task[task_id]
    stdlib_random.shuffle(items)
    split_idx = max(1, int(len(items) * 0.9))
    train_items.extend(items[:split_idx])
    eval_items.extend(items[split_idx:])

stdlib_random.shuffle(train_items)
stdlib_random.shuffle(eval_items)

# Store as plain text — no nested dicts, no serialization issues
train_ds = Dataset.from_list([{"text": item["text"]} for item in train_items])
eval_ds = Dataset.from_list([{"text": item["text"]} for item in eval_items])

print(f"\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Eval task distribution: {Counter(item['task_id'] for item in eval_items)}")
print(f"Format: {'Think-then-Act (CoT)' if USE_COT else 'Plain JSON'}")
print(f"Observation window: {OBS_MAX_CHARS} chars")
print(f"\nSample text (first 300 chars):")
print(f"  {train_ds[0]['text'][:300]}...")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Found CoT-augmented data — using Think-then-Act format for SFT
  Loaded 250 CoT-augmented records

Train: 224 | Eval: 26
Eval task distribution: Counter({2: 13, 1: 13})
Format: Think-then-Act (CoT)
Observation window: 3500 chars

Sample text (first 300 chars):
  <|im_start|>system
You are a web agent navigating an interactive web application.
You observe an accessibility tree and must complete a booking task.

First, reason about what you see and what action to take inside <think> tags.
Then, provide your action as a JSON object inside <answer> tags.

Forma...


In [11]:
# Cell 5 — Load base model with Unsloth (4-bit QLoRA)
# Guide §10: TRL + Unsloth for efficiency
# Guide §16: Use QLoRA, save adapters properly

from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"  # Small enough for T4, capable enough for tasks
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # Auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit=True, # QLoRA — fits in T4 16GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,              # LoRA rank — low rank limits capacity, prevents overfitting
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,    # Unsloth recommends 0 for speed
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {MODEL_ID}")
print(f"Trainable params: {model.print_trainable_parameters()}")

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded: unsloth/Qwen2.5-3B-Instruct
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Trainable params: None


In [12]:
# Cell 6 — SFT warmup training with early stopping
# Guide §3: SFT first to give the model a warm start before RL
#
# Data is already pre-formatted as plain text strings in Cell 4
# (using tokenizer.apply_chat_template), so no formatting_func needed.
#
# Anti-overfitting: EarlyStoppingCallback stops if eval loss plateaus.

from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
import math

# Calculate steps to set good logging/eval intervals
num_samples = len(train_ds)
batch_size = 4
grad_accum = 4
effective_batch = batch_size * grad_accum
steps_per_epoch = math.ceil(num_samples / effective_batch)
total_steps = steps_per_epoch * 3  # 3 epochs

# Set eval/log intervals proportional to dataset size
# Aim for ~8-10 eval points across training
eval_interval = max(1, steps_per_epoch // 3)  # ~3 evals per epoch = ~9 total
log_interval = max(1, eval_interval // 2)     # Log twice per eval interval

print(f"Dataset: {num_samples} samples")
print(f"Steps per epoch: {steps_per_epoch} | Total steps: {total_steps}")
print(f"Eval every {eval_interval} steps | Log every {log_interval} steps")
print(f"Expected eval points: ~{total_steps // eval_interval}")
print()

sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    warmup_steps=min(50, total_steps // 5),
    logging_steps=log_interval,
    save_steps=eval_interval,
    eval_strategy="steps",
    eval_steps=eval_interval,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Starting SFT training...")
print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"  Epochs: 3 | Early stopping patience: 3 eval checks")
print()

sft_results = trainer.train()
print(f"\nSFT complete!")
print(f"  Final training loss: {sft_results.training_loss:.4f}")
if hasattr(trainer.state, "best_metric") and trainer.state.best_metric is not None:
    print(f"  Best eval loss: {trainer.state.best_metric:.4f}")

Dataset: 224 samples
Steps per epoch: 14 | Total steps: 42
Eval every 4 steps | Log every 2 steps
Expected eval points: ~10



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/224 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/26 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting SFT training...
  Train: 224 | Eval: 26
  Epochs: 3 | Early stopping patience: 3 eval checks



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 224 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
4,1.359971,1.140361
8,0.974649,0.855048
12,0.713962,0.627920
16,0.519707,0.429401
20,0.315395,0.251550
24,0.175357,0.130583
28,0.092425,0.068691
32,0.053971,0.044951
36,0.039707,0.034982
40,0.032691,0.030229


Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-12/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-32/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-36/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sft_output/checkpoint-40/tokenizer_confi


SFT complete!
  Final training loss: 0.4391
  Best eval loss: 0.0295


In [13]:
# Cell 7 — Save SFT checkpoint to Google Drive + sanity check
#
# Saves to Google Drive so you can resume in a new session if T4 crashes.
# Next time, skip Cells 5-7 and run Cell 7.5 (Resume) instead.

import gc

# ── Save to Google Drive (survives session crashes) ──
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/infinite-dom-checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

SFT_SAVE_PATH = f"{DRIVE_CHECKPOINT_DIR}/sft_lora_adapters"

model.save_pretrained(SFT_SAVE_PATH)
tokenizer.save_pretrained(SFT_SAVE_PATH)
print(f"SFT adapters saved to Google Drive: {SFT_SAVE_PATH}")

# Also save locally for immediate use
model.save_pretrained("./sft_lora_adapters")
tokenizer.save_pretrained("./sft_lora_adapters")
print("SFT adapters also saved locally: ./sft_lora_adapters")

# ── Free SFT trainer memory ──
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"Freed SFT trainer memory. GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# ── Sanity check ──
def parse_model_output(text):
    """Extract JSON action from <answer> tags or raw JSON."""
    text = text.strip()
    if "<answer>" in text:
        start = text.index("<answer>") + len("<answer>")
        end = text.index("</answer>") if "</answer>" in text else len(text)
        text = text[start:end].strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text)

FastLanguageModel.for_inference(model)

test_observations = [
    """Task: Book a Sleeper ticket from Delhi to Mumbai

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Trains"]
    [ref=inp_1 role=textbox name="From" value=""]
    [ref=inp_2 role=textbox name="To" value=""]
    [ref=cmb_1 role=combobox name="Class" value="-- Select --"]
    [ref=btn_1 role=button name="Search"]""",

    """Task: Book a Chair Car ticket from Pune to Jaipur

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=btn_2 role=button name="Accept" description="cookie banner"]
  [ref=frm_1 role=form name="Booking"]
    [ref=cmb_1 role=combobox name="Seat Type" value="-- Choose --"]
    [ref=inp_1 role=textbox name="Origin" value=""]
    [ref=inp_2 role=textbox name="Destination" value=""]
    [ref=btn_1 role=button name="Go"]""",
]

print("\nSFT Sanity Check:")
print("=" * 60)

for i, test_input in enumerate(test_observations):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_input},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"\nTest {i+1} ({len(response.split())} tokens):")
    print(f"  {response.strip()[:250]}")

    try:
        parsed = parse_model_output(response)
        print(f"  Parsed: {parsed}")
        print(f"  Status: PASS")
    except Exception:
        print(f"  Status: FAIL (could not parse JSON)")

print(f"\n{'=' * 60}")
print("If PASS: proceed to Cell 8 (GRPO)")
print("If FAIL: check Cell 4 data format or increase SFT epochs")

Mounted at /content/drive


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/infinite-dom-checkpoints/sft_lora_adapters/tokenizer_config.json.


SFT adapters saved to Google Drive: /content/drive/MyDrive/infinite-dom-checkpoints/sft_lora_adapters


Unsloth: Restored added_tokens_decoder metadata in ./sft_lora_adapters/tokenizer_config.json.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SFT adapters also saved locally: ./sft_lora_adapters
Freed SFT trainer memory. GPU free: 10.4 GB

SFT Sanity Check:


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Test 1 (11 tokens):
  <think>
I see a cookie banner. Clicking "Accept" first.
</think>
{"action_type":"click","element_ref":"bnr_1","text_value":"","scroll_delta":0}
  Status: FAIL (could not parse JSON)

Test 2 (18 tokens):
  <think>
I've filled cmb_1="Chair Car", inp_1="Pune", and inp_2="Jaipur". Clicking btn_1="Go" to submit the search.
</think>
<answer>
{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}
</answer>
  Parsed: {'action_type': 'click', 'element_ref': 'btn_1', 'text_value': '', 'scroll_delta': 0}
  Status: PASS

If PASS: proceed to Cell 8 (GRPO)
If FAIL: check Cell 4 data format or increase SFT epochs


In [14]:
# Cell 8 — GRPO RL Training (Final Version)
#
# Features:
#   - Self-contained reward (no broken oracle lookup)
#   - Saves to Google Drive AFTER EACH TASK (crash-safe)
#   - Skips already-completed tasks (resume-friendly)
#   - Memory cleanup between tasks (prevents T4 OOM)
#   - T4-optimized config (lighter than before)

from trl import GRPOTrainer, GRPOConfig
import re
import gc

FastLanguageModel.for_training(model)

SCORE_MIN, SCORE_MAX = 0.01, 0.99
VALID_ACTION_TYPES = {"click", "type", "scroll", "wait"}
VALID_REF_PATTERN = re.compile(r"^(inp|btn|cmb|lnk|frm|hdg|mn|nav|chk|rad|opt)_\d+$")

KNOWN_CITIES = {
    "bengaluru", "mumbai", "delhi", "chennai", "kolkata",
    "hyderabad", "pune", "ahmedabad", "jaipur", "lucknow",
}
KNOWN_CLASSES = {
    "sleeper", "ac 3 tier", "ac 2 tier", "ac 1st class", "chair car",
}

DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/infinite-dom-checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# Check which tasks were already completed (from Cell 7.5 resume)
if "COMPLETED_TASKS" not in dir():
    COMPLETED_TASKS = set()


def completion_quality_reward(completions, **kwargs):
    """
    Self-contained reward — scores ONLY from the completion text.

    D1: Reasoning Depth    (0.00 - 0.35) ← main variance driver
    D2: Content Grounding  (0.00 - 0.30) ← real cities/refs vs hallucinated
    D3: Coherence          (0.00 - 0.20) ← reasoning matches action
    D4: Format & Structure (0.00 - 0.14) ← baseline hygiene

    Lazy output: ~0.18 | Good output: ~0.55 | Excellent: ~0.85+
    """
    rewards = []

    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        text = text.strip()

        d1 = d2 = d3 = d4 = 0.0

        # ── Extract think and answer blocks ──
        think_text = ""
        has_think = "<think>" in text and "</think>" in text
        has_answer = "<answer>" in text

        if has_think:
            try:
                t_s = text.index("<think>") + 7
                t_e = text.index("</think>")
                think_text = text[t_s:t_e].strip()
            except ValueError:
                pass

        action_text = text
        if has_answer:
            try:
                a_s = text.index("<answer>") + 8
                a_e = text.index("</answer>") if "</answer>" in text else len(text)
                action_text = text[a_s:a_e].strip()
            except ValueError:
                pass
        if action_text.startswith("```"):
            parts = action_text.split("```")
            if len(parts) > 1:
                action_text = parts[1].lstrip("json").strip()

        try:
            predicted = json.loads(action_text)
        except (json.JSONDecodeError, TypeError):
            predicted = {}

        if not isinstance(predicted, dict):
            rewards.append(SCORE_MIN)
            continue

        atype = predicted.get("action_type", "")
        ref = predicted.get("element_ref", "")
        tval = predicted.get("text_value", "")

        # ── D1: REASONING DEPTH (max 0.35) ──
        if has_think and has_answer:
            d1 += 0.04
            think_len = len(think_text)
            if think_len < 20:
                d1 += 0.00
            elif think_len < 60:
                d1 += 0.04
            elif think_len < 120:
                d1 += 0.10
            elif think_len < 200:
                d1 += 0.16
            else:
                d1 += 0.20

            think_refs = re.findall(r"(inp|btn|cmb|lnk)_\d+", think_text.lower())
            if len(think_refs) >= 2:
                d1 += 0.06
            elif len(think_refs) >= 1:
                d1 += 0.03

            task_words = sum(1 for w in ("field", "button", "form", "fill", "select",
                                          "search", "book", "confirm", "empty", "click",
                                          "type", "origin", "destination", "class", "ticket")
                            if w in think_text.lower())
            if task_words >= 4:
                d1 += 0.05
            elif task_words >= 2:
                d1 += 0.02
        elif has_answer:
            d1 += 0.02

        # ── D2: CONTENT GROUNDING (max 0.30) ──
        if atype in VALID_ACTION_TYPES:
            d2 += 0.05
        if ref and VALID_REF_PATTERN.match(ref):
            d2 += 0.06
        elif ref:
            d2 += 0.01

        if atype == "type" and tval:
            tval_lower = tval.lower().strip()
            if tval_lower in KNOWN_CITIES:
                d2 += 0.12
            elif tval_lower in KNOWN_CLASSES:
                d2 += 0.10
            elif len(tval) > 2 and tval[0].isupper():
                d2 += 0.04
            elif tval:
                d2 += 0.01
        elif atype == "click":
            if ref.startswith("btn_"):
                d2 += 0.07
            elif ref.startswith("lnk_"):
                d2 += 0.05
            elif ref.startswith("inp_"):
                d2 += 0.01

        # ── D3: COHERENCE (max 0.20) ──
        if think_text and predicted:
            think_lower = think_text.lower()
            if ref and ref.lower() in think_lower:
                d3 += 0.08
            if tval and tval.lower() in think_lower:
                d3 += 0.06
            if atype == "type" and any(w in think_lower for w in ("type", "fill", "enter", "input", "write")):
                d3 += 0.06
            elif atype == "click" and any(w in think_lower for w in ("click", "press", "tap", "submit", "dismiss")):
                d3 += 0.06
            elif atype == "scroll" and any(w in think_lower for w in ("scroll", "look", "see more")):
                d3 += 0.06
            elif atype == "wait" and any(w in think_lower for w in ("wait", "loading", "pause")):
                d3 += 0.06

        # ── D4: FORMAT & STRUCTURE (max 0.14) ──
        if atype in VALID_ACTION_TYPES:
            d4 += 0.06
        if atype in ("click", "type") and ref:
            d4 += 0.04
        elif atype in ("scroll", "wait"):
            d4 += 0.04
        expected_keys = {"action_type", "element_ref", "text_value", "scroll_delta"}
        if predicted and set(predicted.keys()).issubset(expected_keys):
            d4 += 0.04

        total = max(SCORE_MIN, min(SCORE_MAX, d1 + d2 + d3 + d4))
        rewards.append(total)

    return rewards


# ─── T4-Optimized Config ───
grpo_config = GRPOConfig(
    output_dir="./grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    logging_steps=5,
    save_steps=100,
    max_completion_length=200,
    num_generations=2,
    temperature=0.8,
    report_to="none",
    seed=42,
)

# ─── Curriculum training with per-task checkpointing ───
reward_history = []
PROMPTS_PER_TASK = 50

for task_id in [1, 2, 3, 4]:
    # Skip already-completed tasks (for resume)
    if task_id in COMPLETED_TASKS:
        print(f"\n>>> Task {task_id}: ALREADY DONE (loaded from Drive checkpoint) — skipping")
        continue

    print(f"\n{'='*55}")
    print(f"GRPO Training — Task {task_id}")
    print(f"{'='*55}")

    task_records = [r for r in records if r["task_id"] == task_id]
    if not task_records:
        print(f"  No oracle data for task {task_id} — skipping")
        continue

    task_records = task_records[:PROMPTS_PER_TASK]

    prompts = []
    for r in task_records:
        obs_text = r["observation"][:OBS_MAX_CHARS]
        step = r.get("step", 0)
        step_ctx = f"\nStep: {step}" if step > 0 else ""
        user_msg = f"Task: {r['instruction']}\n\nAccessibility Tree:\n{obs_text}{step_ctx}"
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        prompts.append(msgs)

    prompt_dataset = Dataset.from_dict({"prompt": prompts})

    total_steps = len(prompts) // (grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps)
    print(f"  Prompts: {len(prompts)} | Steps: ~{total_steps}")
    print(f"  LR: {grpo_config.learning_rate:.2e} | Temp: {grpo_config.temperature}")
    print(f"  GPU free before: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=[completion_quality_reward],
        train_dataset=prompt_dataset,
        args=grpo_config,
    )

    result = grpo_trainer.train()

    # ── Extract metrics ──
    avg_reward = None
    final_reward_std = None
    if hasattr(grpo_trainer, "state") and grpo_trainer.state.log_history:
        reward_logs = [h["reward"] for h in grpo_trainer.state.log_history if "reward" in h]
        std_logs = [h["reward_std"] for h in grpo_trainer.state.log_history if "reward_std" in h]
        if reward_logs:
            avg_reward = sum(reward_logs) / len(reward_logs)
        if std_logs:
            final_reward_std = std_logs[-1]

    loss_val = result.training_loss
    print(f"\n  Results:")
    print(f"    Training loss: {loss_val:.4f}")
    if avg_reward is not None:
        print(f"    Avg reward: {avg_reward:.4f}")
    if final_reward_std is not None:
        health = "HEALTHY" if final_reward_std > 0.03 else "LOW" if final_reward_std > 0.01 else "COLLAPSED"
        print(f"    Final reward_std: {final_reward_std:.4f} ({health})")

    reward_history.append({
        "task_id": task_id,
        "loss": loss_val,
        "avg_reward": avg_reward,
        "final_std": final_reward_std,
    })

    # ── Save checkpoint to Google Drive (crash-safe) ──
    save_path = f"{DRIVE_CHECKPOINT_DIR}/grpo_after_task_{task_id}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    COMPLETED_TASKS.add(task_id)
    print(f"    Saved to Drive: {save_path}")

    # ── Free memory before next task ──
    del grpo_trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"    GPU free after cleanup: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    # Decay LR for harder tasks
    grpo_config.learning_rate *= 0.7

print(f"\n{'='*55}")
print("GRPO curriculum training complete!")
print(f"{'='*55}")
print(f"\nWhat to check:")
print(f"  reward should INCREASE across steps (model improving)")
print(f"  reward_std should stay > 0.03 (GRPO has signal)")
print(f"  mean_length should be > 80 tokens (model is reasoning)")
print()
for rh in reward_history:
    r_str = f"  reward={rh['avg_reward']:.4f}" if rh.get("avg_reward") else ""
    s_str = f"  std={rh['final_std']:.4f}" if rh.get("final_std") else ""
    print(f"  Task {rh['task_id']}: loss={rh['loss']:.4f}{r_str}{s_str}")


GRPO Training — Task 1
  Prompts: 50 | Steps: ~12
  LR: 5.00e-06 | Temp: 0.8
  GPU free before: 10.2 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 1 | Total steps = 25
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWa

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / completion_quality_reward / mean,rewards / completion_quality_reward / std
5,0.001648,0.630500,0.074246,69.550000,56.000000,98.000000,0.050000,62.616667,56.000000,72.000000,1.647558,0.630500,0.105000
10,0.001796,0.652500,0.043134,62.750000,57.800000,74.400000,0.000000,62.750000,57.800000,74.400000,1.795813,0.652500,0.059485
15,0.001653,0.683000,0.008485,73.000000,60.600000,96.200000,0.050000,66.383334,60.600000,70.800000,1.653188,0.683000,0.012000
20,0.001697,0.684500,0.021920,69.600000,59.200000,94.400000,0.050000,63.216667,59.200000,69.000000,1.696934,0.684500,0.025839
25,0.001737,0.663000,0.045255,64.350000,59.000000,70.800000,0.000000,64.350000,59.000000,70.800000,1.736728,0.663000,0.057013


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

...

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / completion_quality_reward / mean,rewards / completion_quality_reward / std
5,0.001720,0.639500,0.058690,67.400000,58.000000,90.000000,0.000000,67.400000,58.000000,90.000000,1.720772,0.639500,0.079061
10,0.001597,0.631000,0.041012,60.500000,56.400000,68.600000,0.000000,60.500000,56.400000,68.600000,1.597564,0.631000,0.072038
15,0.001745,0.659500,0.037477,61.450000,53.600000,74.600000,0.000000,61.450000,53.600000,74.600000,1.745773,0.659500,0.051685
20,0.001617,0.682000,0.014142,60.000000,57.400000,66.600000,0.000000,60.000000,57.400000,66.600000,1.617291,0.682000,0.016899
25,0.001580,0.678500,0.024749,61.550000,57.000000,70.400000,0.000000,61.550000,57.000000,70.400000,1.579649,0.678500,0.032406


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


  Results:
    Training loss: 0.0017
    Avg reward: 0.6581
    Final reward_std: 0.0247 (LOW)


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/infinite-dom-checkpoints/grpo_after_task_2/tokenizer_config.json.


    Saved to Drive: /content/drive/MyDrive/infinite-dom-checkpoints/grpo_after_task_2
    GPU free after cleanup: 10.3 GB

GRPO Training — Task 3
  No oracle data for task 3 — skipping

GRPO Training — Task 4
  No oracle data for task 4 — skipping

GRPO curriculum training complete!

What to check:
  reward should INCREASE across steps (model improving)
  reward_std should stay > 0.03 (GRPO has signal)
  mean_length should be > 80 tokens (model is reasoning)

  Task 1: loss=0.0017  reward=0.6627  std=0.0453
  Task 2: loss=0.0017  reward=0.6581  std=0.0247


In [15]:
# Cell 9 — Post-RL Quality Check: reasoning + diversity + correctness
#
# Checks from WebAgent-R1 insights:
# 1. Does the model produce <think>...</think><answer>...</answer> format?
# 2. Is the reasoning relevant (mentions elements, task terms)?
# 3. Are actions still valid JSON?
# 4. Is there diversity (no mode collapse)?

FastLanguageModel.for_inference(model)

quality_tests = [
    ("Easy", """Task: Book a Sleeper ticket from Delhi to Mumbai

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Trains"]
    [ref=inp_1 role=textbox name="From" value=""]
    [ref=inp_2 role=textbox name="To" value=""]
    [ref=cmb_1 role=combobox name="Class" value="-- Select --"]
    [ref=btn_1 role=button name="Search"]"""),

    ("Medium", """Task: Book an AC 2 Tier ticket from Hyderabad to Pune

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Journey Planner"]
    [ref=inp_1 role=textbox name="Starting Point" value=""]
    [ref=inp_2 role=textbox name="Going To" value=""]
    [ref=cmb_1 role=combobox name="Coach Class" value="-- Pick --"]
    [ref=btn_1 role=button name="Check Availability"]"""),

    ("Hard", """Task: Book a Chair Car ticket from Kolkata to Lucknow

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=btn_99 role=button name="Accept" description="cookie banner"]
  [ref=sec_1 role=region name="Special Offer"]
    [ref=btn_98 role=button name="Claim Now"]
  [ref=frm_1 role=form name="Travel Booking"]
    [ref=cmb_1 role=combobox name="Seat Type" value="-- Choose --"]
    [ref=inp_1 role=textbox name="Origin" value=""]
    [ref=inp_2 role=textbox name="Destination" value=""]
    [ref=btn_1 role=button name="Look Up Trains"]"""),
]

print("Post-RL Quality Check (Think-then-Act)")
print("=" * 60)

stats = {"json_valid": 0, "action_valid": 0, "has_think": 0, "total": 0}
all_outputs = []

for difficulty, test_input in quality_tests:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_input},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs_for_input = []
    for _ in range(3):
        outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.3, do_sample=True)
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        outputs_for_input.append(response)
        all_outputs.append(response)
        stats["total"] += 1

        if "<think>" in response:
            stats["has_think"] += 1
        try:
            parsed = parse_model_output(response)
            stats["json_valid"] += 1
            if parsed.get("action_type") in VALID_ACTION_TYPES:
                stats["action_valid"] += 1
        except (json.JSONDecodeError, ValueError):
            pass

    unique = len(set(outputs_for_input))
    print(f"\n[{difficulty}]")
    for j, out in enumerate(outputs_for_input):
        # Show thinking preview + action
        think_preview = ""
        if "<think>" in out and "</think>" in out:
            t_start = out.index("<think>") + 7
            t_end = out.index("</think>")
            think_preview = out[t_start:t_end].strip()[:80] + "..."
        action_preview = ""
        if "<answer>" in out:
            a_start = out.index("<answer>") + 8
            action_preview = out[a_start:a_start+80].strip()
        print(f"  {j+1}: think='{think_preview}' action='{action_preview}'")
    print(f"  Diversity: {unique}/3 unique")

t = stats["total"]
unique_total = len(set(all_outputs))
print(f"\n{'=' * 60}")
print(f"Overall: {stats['json_valid']}/{t} valid JSON, {stats['action_valid']}/{t} valid actions")
print(f"Reasoning: {stats['has_think']}/{t} include <think> tags")
print(f"Diversity: {unique_total}/{t} unique outputs")

if unique_total <= 2:
    print("WARNING: Possible mode collapse.")
elif stats["json_valid"] < t * 0.5:
    print("WARNING: JSON validity dropped after RL.")
else:
    print("Model looks healthy after RL training.")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Post-RL Quality Check (Think-then-Act)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=3


[Easy]
  1: think='I've filled both From and To fields. Clicking "Search" (btn_1) first....' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  2: think='I've filled both From and To fields. Clicking "Search" (btn_1) submits the form....' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  3: think='I've filled both From and To fields. Clicking "Search" (btn_1) first....' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  Diversity: 2/3 unique


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Medium]
  1: think='I've filled both textboxes (Starting Point & Going To). Clicking "Go" (btn_3) is...' action='{"action_type":"click","element_ref":"btn_3","text_value":"","scroll_delta":0}'
  2: think='I've filled both textboxes (Starting Point & Going To). Clicking "Go" (btn_2) is...' action='{"action_type":"click","element_ref":"btn_2","text_value":"","scroll_delta":0}'
  3: think='I've filled both textboxes (Starting Point & Going To). Clicking "Go" (btn_2) is...' action='{"action_type":"click","element_ref":"btn_2","text_value":"","scroll_delta":0}'
  Diversity: 2/3 unique


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Hard]
  1: think='I've filled cmb_1="Chair Car", inp_1="Kolkata", inp_2="Lucknow". Clicking btn_1 ...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  2: think='I've filled cmb_1="Chair Car", inp_1="Kolkata", inp_2="Lucknow". Clicking btn_1 ...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  3: think='I've filled cmb_1="Chair Car", inp_1="Kolkata", inp_2="Lucknow". Clicking btn_1 ...' action='{"action_type":"click","element_ref":"btn_1","text_value":"","scroll_delta":0}'
  Diversity: 1/3 unique

Overall: 9/9 valid JSON, 9/9 valid actions
Reasoning: 9/9 include <think> tags
Diversity: 5/9 unique outputs
Model looks healthy after RL training.


In [16]:
# Cell 10 — Live Environment Evaluation with Step History
#
# WebAgent-R1 insights applied:
#   1. Dynamic context compression: pass compressed step history to the model
#   2. Test-time scaling: allow up to 30 steps (they showed more steps = better)
#   3. Think-then-Act format for inference

import httpx

INFINITE_DOM_URL = os.environ.get("INFINITE_DOM_URL", HF_SPACE_URL)

print("Checking environment connection...")
try:
    r = httpx.get(f"{INFINITE_DOM_URL}/health", timeout=30)
    if r.status_code != 200:
        raise ConnectionError(f"Health check failed: HTTP {r.status_code}")
    print(f"Connected to: {INFINITE_DOM_URL}")
except Exception as e:
    print(f"ERROR: Cannot connect to environment: {e}")
    print("Skip this cell and run it later when the HF Space is available.")
    raise SystemExit("Environment not available")


MAX_EVAL_STEPS = 30  # WebAgent-R1: more interactions = better performance


def run_eval_episode(policy_fn, task_id=1, seed=42, max_steps=MAX_EVAL_STEPS):
    """Run one full episode against the live environment."""
    for attempt in range(3):
        try:
            r = httpx.post(
                f"{INFINITE_DOM_URL}/reset",
                json={"task_id": task_id, "seed": seed},
                timeout=120,
            )
            obs = r.json()
            break
        except (httpx.TimeoutException, httpx.ConnectError) as e:
            if attempt == 2:
                print(f"  Reset failed after 3 attempts: {e}")
                return 0, 0.0, 0
            print(f"  Reset attempt {attempt+1} failed, retrying...")
            time.sleep(5)

    total_reward = 0.0
    steps = 0
    step_history = []  # Dynamic context compression buffer

    for step in range(max_steps):
        if obs.get("done", False):
            break
        action = policy_fn(obs, step_history)
        try:
            r = httpx.post(
                f"{INFINITE_DOM_URL}/step",
                json={"action": action},
                timeout=30,
            )
            obs = r.json()
            total_reward += obs.get("reward", 0) or 0
            steps = step + 1

            # Record step for context compression
            atype = action.get("action_type", "?")
            ref = action.get("element_ref", "")
            tval = action.get("text_value", "")
            succeeded = obs.get("metadata", {}).get("action_succeeded", True)
            desc = f"Step {steps}: {atype}"
            if ref:
                desc += f" on {ref}"
            if tval:
                desc += f" ('{tval}')"
            desc += f" {'ok' if succeeded else 'FAILED'}"
            step_history.append(desc)

        except Exception as e:
            print(f"  Step {step} failed: {e}")
            break

    completed = len(obs.get("task_progress", []))
    return completed, total_reward, steps


def random_policy(obs, step_history=None):
    import random as rng
    return {
        "action_type": rng.choice(["click", "type", "wait"]),
        "element_ref": "btn_1",
        "text_value": "test",
        "scroll_delta": 0,
    }


def model_policy(obs, step_history=None):
    """Use trained model with dynamic context compression (WebAgent-R1 style)."""
    tree = obs.get("a11y_tree", "")[:OBS_MAX_CHARS]
    instruction = obs.get("task_instruction", "")
    progress = obs.get("task_progress", [])

    # Build compressed step history (last 5 steps)
    history_text = ""
    if step_history:
        recent = step_history[-5:]  # Keep only recent context
        history_text = "Previous actions:\n" + "\n".join(f"  {s}" for s in recent) + "\n\n"

    user_msg = (
        f"Task: {instruction}\n\n"
        f"{history_text}"
        f"Accessibility Tree:\n{tree}\n\n"
        f"Completed: {progress}"
    )

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        return parse_model_output(response)
    except (json.JSONDecodeError, ValueError):
        return {"action_type": "wait", "element_ref": "", "text_value": "", "scroll_delta": 0}


# --- Run evaluations ---
print(f"\nMax steps per episode: {MAX_EVAL_STEPS} (WebAgent-R1: more interactions = better)")

print("\n=== Random Baseline ===")
random_results = {}
for task_id in [1, 2]:
    nodes_list = []
    for seed in range(3):
        nodes, reward, steps = run_eval_episode(random_policy, task_id=task_id, seed=seed)
        nodes_list.append(nodes)
    avg = sum(nodes_list) / len(nodes_list)
    random_results[task_id] = avg
    print(f"  Task {task_id}: avg {avg:.1f}/5 nodes completed")

print("\n=== Trained Model (Think-then-Act + Step History) ===")
trained_results = {}
for task_id in [1, 2, 3, 4]:
    nodes_list = []
    steps_list = []
    for seed in range(5):
        nodes, reward, steps = run_eval_episode(model_policy, task_id=task_id, seed=seed)
        nodes_list.append(nodes)
        steps_list.append(steps)
    avg = sum(nodes_list) / len(nodes_list)
    avg_steps = sum(steps_list) / len(steps_list)
    trained_results[task_id] = avg
    print(f"  Task {task_id}: avg {avg:.1f}/5 nodes, {avg_steps:.0f} steps")

print("\n=== Summary ===")
random_avg = sum(random_results.values()) / max(len(random_results), 1)
trained_avg = sum(trained_results.values()) / len(trained_results)
print(f"Random baseline: ~{random_avg:.1f} nodes avg")
print(f"Trained model:   {trained_avg:.1f} nodes avg")
print(f"Improvement:     +{trained_avg - random_avg:.1f} nodes")
print("\nInclude these numbers in your README and blog post.")

Checking environment connection...
Connected to: https://saksham1771-infinite-dom.hf.space

Max steps per episode: 30 (WebAgent-R1: more interactions = better)

=== Random Baseline ===
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Task 1: avg 0.0/5 nodes completed
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Task 2: avg 0.0/5 nodes completed

=== Trained Model (Think-then-Act + Step History) ===


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 0 failed: Expecting value: line 1 column 1 (char 0)


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 0 failed: Expecting value: line 1 column 1 (char 0)


...

  Step 0 failed: Expecting value: line 1 column 1 (char 0)


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 0 failed: Expecting value: line 1 column 1 (char 0)


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Step 0 failed: Expecting value: line 1 column 1 (char 0)
  Task 4: avg 0.0/5 nodes, 0 steps

=== Summary ===
Random baseline: ~0.0 nodes avg
Trained model:   0.0 nodes avg
Improvement:     +0.0 nodes

Include these numbers in your README and blog post.


In [ ]:
# Cell 11 — Save final model (Guide §16: save LoRA adapters properly)
#
# WARNING: Do NOT upcast 4-bit to 16-bit and merge naively.
# Save LoRA adapters separately — this is the safe, recommended path.

import os

# Save adapters locally
model.save_pretrained("./final_lora_adapters")
tokenizer.save_pretrained("./final_lora_adapters")
print("Saved final LoRA adapters to ./final_lora_adapters")

# Save eval results alongside the model
eval_results = {
    "sft_training_loss": sft_results.training_loss,
    "reward_history": reward_history,
    "random_baseline": random_results if "random_results" in dir() else {},
    "trained_results": trained_results if "trained_results" in dir() else {},
    "model_id": MODEL_ID,
    "obs_max_chars": OBS_MAX_CHARS,
}
with open("./final_lora_adapters/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print("Saved eval_results.json alongside adapters")

# Push to HuggingFace Hub
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_REPO = "YOUR_USERNAME/infinite-dom-agent"  # @param {type:"string"}

if HF_TOKEN:
    try:
        model.push_to_hub(HF_REPO, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
        # Also push eval results
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj="./final_lora_adapters/eval_results.json",
            path_in_repo="eval_results.json",
            repo_id=HF_REPO,
        )
        print(f"Pushed to HuggingFace Hub: {HF_REPO}")
    except Exception as e:
        print(f"Hub push failed: {e}")
        print("Model saved locally — you can push manually later.")
else:
    print("Set HF_TOKEN to push to Hub. Model saved locally.")

In [ ]:
# Cell 12 — Training Plots (save as PNG for README/blog post)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 150

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: SFT Training + Eval Loss
if hasattr(trainer, "state") and trainer.state.log_history:
    train_losses = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h and "eval_loss" not in h]
    eval_losses = [(h["step"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]

    if train_losses:
        steps, losses = zip(*train_losses)
        axes[0].plot(steps, losses, "b-", linewidth=1.2, alpha=0.7, label="Train")
    if eval_losses:
        steps, losses = zip(*eval_losses)
        axes[0].plot(steps, losses, "r-", linewidth=1.5, label="Eval")
    axes[0].set_xlabel("Training Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("SFT Training & Eval Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Check for overfitting: if eval loss rises while train loss falls
    if eval_losses and len(eval_losses) > 3:
        last_3_eval = [l for _, l in eval_losses[-3:]]
        if last_3_eval[-1] > last_3_eval[0]:
            axes[0].annotate("Potential\noverfitting",
                           xy=(eval_losses[-1][0], eval_losses[-1][1]),
                           fontsize=8, color="red", ha="right")
else:
    axes[0].text(0.5, 0.5, "No SFT data\n(run Cell 6 first)", ha="center", va="center", transform=axes[0].transAxes)

# Plot 2: GRPO Reward by Task (Curriculum)
if reward_history:
    task_ids = [h["task_id"] for h in reward_history]
    rewards = [h.get("avg_reward") or 0 for h in reward_history]
    colors = ["#22c55e", "#eab308", "#f97316", "#ef4444"][:len(task_ids)]
    bars = axes[1].bar(task_ids, rewards, color=colors, edgecolor="black", linewidth=0.5)

    axes[1].set_xlabel("Task ID")
    axes[1].set_ylabel("Avg Reward")
    axes[1].set_title("GRPO Avg Reward by Task")
    axes[1].set_xticks(task_ids)
    labels = ["Clean\nForm", "Label\nDrift", "Structural\nDrift", "Full\nChaos"][:len(task_ids)]
    axes[1].set_xticklabels(labels)
    axes[1].set_ylim(0, 1.0)
    axes[1].grid(True, alpha=0.3, axis="y")

    for bar, val in zip(bars, rewards):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=9)
else:
    axes[1].text(0.5, 0.5, "No GRPO data\n(run Cell 8 first)", ha="center", va="center", transform=axes[1].transAxes)

# Plot 3: Before/After Comparison
if "trained_results" in dir() and trained_results:
    task_ids = sorted(trained_results.keys())
    trained_vals = [trained_results[t] for t in task_ids]
    random_vals = [random_results.get(t, 0) for t in task_ids]

    x = range(len(task_ids))
    width = 0.35
    axes[2].bar([i - width/2 for i in x], random_vals, width, label="Random", color="#94a3b8", edgecolor="black", linewidth=0.5)
    axes[2].bar([i + width/2 for i in x], trained_vals, width, label="Trained", color="#3b82f6", edgecolor="black", linewidth=0.5)

    axes[2].set_xlabel("Task ID")
    axes[2].set_ylabel("Avg Nodes Completed (out of 5)")
    axes[2].set_title("Random vs Trained Agent")
    axes[2].set_xticks(list(x))
    axes[2].set_xticklabels([str(t) for t in task_ids])
    axes[2].set_ylim(0, 5.5)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3, axis="y")
else:
    axes[2].text(0.5, 0.5, "No eval data\n(run Cell 10 first)", ha="center", va="center", transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png — include in README and blog post")

In [ ]:
# Cell 13 — Summary Report

print("=" * 60)
print("  INFINITE DOM — TRAINING SUMMARY")
print("=" * 60)

print(f"\nBase Model: {MODEL_ID}")
print(f"Quantization: QLoRA 4-bit")
print(f"LoRA Rank: 16")
print(f"Observation Window: {OBS_MAX_CHARS} chars")

print(f"\n--- SFT Phase ---")
print(f"Training loss: {sft_results.training_loss:.4f}")
if hasattr(trainer.state, "best_metric") and trainer.state.best_metric:
    print(f"Best eval loss: {trainer.state.best_metric:.4f}")

print(f"\n--- GRPO Phase ---")
for rh in reward_history:
    r_str = f", avg_reward={rh['avg_reward']:.4f}" if rh["avg_reward"] else ""
    print(f"Task {rh['task_id']}: loss={rh['loss']:.4f}{r_str}")

if "trained_results" in dir() and trained_results:
    print(f"\n--- Live Evaluation ---")
    for tid, avg in sorted(trained_results.items()):
        random_avg = random_results.get(tid, 0)
        delta = avg - random_avg
        print(f"Task {tid}: {avg:.1f}/5 nodes (random: {random_avg:.1f}, improvement: +{delta:.1f})")

print(f"\n--- Submission Checklist ---")
print(f"[ ] HF Space deployed and running")
print(f"[ ] Training notebook runs end-to-end in Colab")
print(f"[ ] LoRA adapters pushed to HF Hub")
print(f"[ ] training_curves.png saved for README/blog")
print(f"[ ] README updated with real results")
print(f"[ ] Blog post written on HuggingFace")
print(f"\n{'=' * 60}")